In [3]:
import numpy as np
import cv2
import librosa
import soundfile as sf

# ============ NEW: pixel value spread across multiple neighboring frequencies ============
def pixel_to_multi_freq(spectrogram, spread=2, kernel="triangular"):
    """
    Forward push-mapping: each source row (pixel) writes into a window of
    (2*spread+1) neighboring output bins instead of exactly one bin.
    Output spans the FULL 0..sr/2 range - nothing gets squeezed into a
    sub-band like frequency_remap does. This just thickens each line;
    it doesn't add new information.
    """
    n_bins, n_frames = spectrogram.shape
    out = np.zeros_like(spectrogram)

    offsets = np.arange(-spread, spread + 1)
    if kernel == "triangular":
        weights = 1.0 - np.abs(offsets) / (spread + 1)
    elif kernel == "box":
        weights = np.ones_like(offsets, dtype=float)
    else:
        raise ValueError("kernel must be 'triangular' or 'box'")
    weights = weights / weights.sum()  # kernel sums to 1 -> smoothing, not amplification

    src_idx = np.arange(n_bins)
    for off, w in zip(offsets, weights):
        dst_idx = src_idx + off
        valid = (dst_idx >= 0) & (dst_idx < n_bins)
        out[dst_idx[valid], :] += w * spectrogram[src_idx[valid], :]

    return out
# ============ END NEW ============


def image_to_audio_multi_freq(image_path, output_audio="output.wav", sr=22050, spread=2, kernel="triangular"):
    # Load image in grayscale
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Image not found or invalid path.")

    # Resize image to match Griffin-Lim constraints (n_fft = 1024, hop_length=512)
    # Height (frequency bins) must be (n_fft // 2 + 1) = 513 for n_fft=1024
    target_height = 513  # For n_fft=1024
    target_width = 128   # Adjust width as needed (e.g., 128 for time steps)
    image = cv2.resize(image, (target_width, target_height))

    # Normalize to [0, 1] and flip vertically (spectrograms use bottom-to-top frequency)
    image_normalized = np.flipud(image.astype(np.float32) / 255.0)

    # Convert image to dB-scaled "spectrogram" (add epsilon to avoid log(0))
    spectrogram_db = librosa.amplitude_to_db(image_normalized + 1e-7, ref=np.max)

    # Convert dB back to linear amplitude
    spectrogram_linear = librosa.db_to_amplitude(spectrogram_db)

    # Griffin-Lim parameters (must match spectrogram dimensions)
    n_fft = 1024
    hop_length = 512
    win_length = 1024

    # ============ NEW: spread each pixel across multiple frequencies ============
    spectrogram_linear = pixel_to_multi_freq(spectrogram_linear, spread=spread, kernel=kernel)
    # ============ END NEW ============

    # Reconstruct audio
    audio = librosa.griffinlim(
        spectrogram_linear,
        n_iter=32,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length
    )

    # Save as WAV file
    sf.write(output_audio, audio, sr)
    print(f"Audio saved to {output_audio}")

# Example usage
image_to_audio_multi_freq(r"D:\Documents\Iquisitionis\103\vehicle-type-recognition\Dataset\Car\Image_32.jpg", "car_multifreq.wav", spread=2)

Audio saved to car_multifreq.wav
